# Reasoning & Task Decomposition
### Practice Notebook

**Assumed pre-installed libraries:** none required (pure Python standard
library).


## 1. Chain-of-thought vs. direct answering

Let's see the *effect* chain-of-thought has, using a task where skipping
intermediate steps is genuinely error-prone: multi-step arithmetic word
problems. Since we don't have a live LLM call, we'll simulate both a
"jump to the answer" stub and a "show your work" stub, and reason about
where each is more or less likely to go wrong.


In [1]:
problem = (
    "A cafe sells filter coffee for Rs.20 and tea for Rs.15. "
    "On Monday they sold 45 filter coffees and 30 teas. "
    "On Tuesday they sold 20% more filter coffees than Monday, and the same "
    "number of teas as Monday. What was the total revenue over both days?"
)

def direct_answer_stub(problem: str) -> str:
    """Simulates an LLM asked to jump straight to a final number with no
    visible intermediate steps. # TODO: replace with a real LLM call
    (system prompt: 'answer with only the final number, no explanation').
    """
    return "(stub) Total revenue: Rs. 2400"   # a plausible-looking but WRONG guess

def chain_of_thought_stub(problem: str) -> str:
    """Simulates an LLM asked to show its work step by step.
    # TODO: replace with a real LLM call
    (prompt: 'think step by step before giving your final answer').
    """
    monday_coffee = 45 * 20
    monday_tea = 30 * 15
    tuesday_coffee_count = 45 * 1.2
    tuesday_coffee = tuesday_coffee_count * 20
    tuesday_tea = 30 * 15
    total = monday_coffee + monday_tea + tuesday_coffee + tuesday_tea
    steps = [
        f"Monday coffee revenue: 45 x Rs.20 = Rs.{monday_coffee}",
        f"Monday tea revenue: 30 x Rs.15 = Rs.{monday_tea}",
        f"Tuesday coffee count: 45 x 1.2 = {tuesday_coffee_count}",
        f"Tuesday coffee revenue: {tuesday_coffee_count} x Rs.20 = Rs.{tuesday_coffee}",
        f"Tuesday tea revenue: 30 x Rs.15 = Rs.{tuesday_tea}",
        f"Total: {monday_coffee} + {monday_tea} + {tuesday_coffee} + {tuesday_tea} = Rs.{total}",
    ]
    return "\n".join(steps)

print("=== Direct answer ===")
print(direct_answer_stub(problem))
print()
print("=== Chain-of-thought ===")
print(chain_of_thought_stub(problem))


=== Direct answer ===
(stub) Total revenue: Rs. 2400

=== Chain-of-thought ===
Monday coffee revenue: 45 x Rs.20 = Rs.900
Monday tea revenue: 30 x Rs.15 = Rs.450
Tuesday coffee count: 45 x 1.2 = 54.0
Tuesday coffee revenue: 54.0 x Rs.20 = Rs.1080.0
Tuesday tea revenue: 30 x Rs.15 = Rs.450
Total: 900 + 450 + 1080.0 + 450 = Rs.2880.0


**Exercise 4.1:** Verify the chain-of-thought answer by hand (or with a
calculator). Was the direct-answer stub's Rs. 2400 correct? This is a
deliberately exaggerated illustration -- a real LLM's "direct" answers
aren't randomly wrong like this stub, but the *mechanism* is real: without
visible intermediate steps, there's no way to check *where* a multi-step
calculation went wrong, for the model or for you as a reviewer.

**Exercise 4.2:** Write your own 3-step word problem (different numbers/
scenario), and manually write out both a `direct_answer_stub`-style guess and
a `chain_of_thought_stub`-style worked solution for it.


Monday revenue
- Filter coffee: \(45 \times 20 = ₹900\)
- Tea: \(30 \times 15 = ₹450\)
- Total Monday revenue: \(₹900 + ₹450 = ₹1,350\)
Tuesday revenue
- Filter coffees sold: \(45 \times 1.2 = 54\)
- Filter coffee revenue: \(54 \times 20 = ₹1,080\)
- Tea revenue: \(30 \times 15 = ₹450\)
- Total Tuesday revenue: \(₹1,080 + ₹450 = ₹1,530\)
Total revenue over both days:
\[
₹1,350 + ₹1,530 = \boxed{₹2,880}
\]

Therefore, the direct-answer stub's ₹2,400 is incorrect. The worked solution gives the correct answer of ₹2,880 and makes each calculation easy to verify.

In [ ]:
#direct answer
def direct_answer_stub(problem: str) -> str:
    return "Total revenue: Rs. 3000"
#worked solution
def chain_of_thought_stub(problem: str) -> str:
    monday_notebooks = 25 * 40
    monday_pens = 40 * 10

    tuesday_notebook_count = 25 * 1.2
    tuesday_notebooks = tuesday_notebook_count * 40
    tuesday_pens = 40 * 10

    total = (
        monday_notebooks + monday_pens
        + tuesday_notebooks + tuesday_pens
    )

    steps = [
        f"Monday notebook revenue: 25 x Rs.40 = Rs.{monday_notebooks}",
        f"Monday pen revenue: 40 x Rs.10 = Rs.{monday_pens}",
        f"Tuesday notebooks sold: 25 x 1.2 = {tuesday_notebook_count}",
        f"Tuesday notebook revenue: Rs.{tuesday_notebooks}",
        f"Tuesday pen revenue: Rs.{tuesday_pens}",
        f"Total revenue: Rs.{total}"
    ]

    return "\n".join(steps)

print("=== Direct answer ===")
print(direct_answer_stub("Bookstore revenue problem"))

print("\n=== Worked solution ===")
print(chain_of_thought_stub("Bookstore revenue problem"))

## 2. Task decomposition

Given a large, ambiguous goal, break it into smaller, concrete sub-tasks
*before* attempting any of them. This connects directly to Day 3's
orchestrator-worker pattern -- decomposition is what the "orchestrator" step
does.


In [2]:
def decompose_task(goal: str) -> list:
    """# TODO: replace with a real LLM call that decomposes `goal` into
    concrete, actionable sub-tasks.
    """
    decomposition_rules = {
        "trip": ["Determine travel dates", "Search flights", "Search hotels",
                 "Build a day-by-day itinerary", "Estimate total budget"],
        "essay": ["Choose a thesis statement", "Outline main arguments",
                  "Write a draft", "Revise for clarity", "Proofread"],
        "app": ["Define core features", "Design the data model",
                "Build the backend", "Build the frontend", "Test end-to-end"],
    }
    for keyword, subtasks in decomposition_rules.items():
        if keyword in goal.lower():
            return subtasks
    return ["Clarify the goal", "Identify needed resources", "Execute", "Review"]

goal = "Plan a weekend trip to Coorg"
subtasks = decompose_task(goal)
print(f"Goal: {goal}")
for i, st in enumerate(subtasks, 1):
    print(f"  {i}. {st}")


Goal: Plan a weekend trip to Coorg
  1. Determine travel dates
  2. Search flights
  3. Search hotels
  4. Build a day-by-day itinerary
  5. Estimate total budget


**Exercise 4.3:** `decompose_task` uses simple keyword matching, so it will
decompose *any* goal containing the word "trip" identically, regardless of
details (a weekend trip vs. a month-long trip should probably decompose
differently -- e.g., a longer trip might need "arrange time off work" or
"apply for a visa"). This is a real limitation of rule-based decomposition
that a real LLM call would handle better by reasoning over the goal's actual
content. Add one new goal category of your own to `decomposition_rules` with
at least 4 sub-tasks.


In [ ]:
def decompose_task(goal: str) -> list:
    """Decompose a goal into actionable sub-tasks."""
    decomposition_rules = {
        "trip": [
            "Determine travel dates",
            "Search flights",
            "Search hotels",
            "Build a day-by-day itinerary",
            "Estimate total budget"
        ],
        "essay": [
            "Choose a thesis statement",
            "Outline main arguments",
            "Write a draft",
            "Revise for clarity",
            "Proofread"
        ],
        "app": [
            "Define core features",
            "Design the data model",
            "Build the backend",
            "Build the frontend",
            "Test end-to-end"
        ],
        "presentation": [
            "Identify the topic and target audience",
            "Research key information",
            "Organize the content into slides",
            "Add visuals and examples",
            "Review and rehearse the presentation"
        ]
    }

    for keyword, subtasks in decomposition_rules.items():
        if keyword in goal.lower():
            return subtasks

    return [
        "Clarify the goal",
        "Identify needed resources",
        "Execute",
        "Review"
    ]


goal = "Prepare a presentation on artificial intelligence"
subtasks = decompose_task(goal)

print(f"Goal: {goal}")
for i, st in enumerate(subtasks, 1):
    print(f"  {i}. {st}")

## 3. Planning strategies: plan-then-execute vs. interleaved

**Plan-then-execute** commits to a full plan up front, then runs it.
**Interleaved planning** (the ReAct pattern from Day 1) plans one step,
observes the real result, and re-plans the next step based on what actually
happened. Let's build both for a task where an early assumption can turn out
to be wrong, so the difference actually matters.


In [3]:
import random

def check_flight_availability(destination: str) -> bool:
    """FAKE tool: randomly reports flights as unavailable ~30% of the time,
    to simulate a real-world surprise a plan might not have accounted for.
    """
    return random.random() > 0.3

def plan_then_execute(destination: str):
    print("--- Plan-then-execute ---")
    full_plan = ["Book flight", "Book hotel", "Plan itinerary"]
    print(f"Full plan committed up front: {full_plan}")
    for step in full_plan:
        if step == "Book flight":
            available = check_flight_availability(destination)
            print(f"Executing '{step}'... available={available}")
            if not available:
                print("  Flight unavailable, but the plan didn't account for "
                      "this -- proceeding anyway (this is the brittleness "
                      "of committing to a full plan up front).")
        else:
            print(f"Executing '{step}'...")

def interleaved_planning(destination: str):
    print("--- Interleaved planning (ReAct-style) ---")
    print("Thought: first step should be booking a flight. Let's check availability.")
    available = check_flight_availability(destination)
    print(f"Observation: flight available = {available}")
    if not available:
        print("Thought: flight unavailable -- re-planning: try a nearby "
              "airport or a different date before booking a hotel.")
        print("Action: check_flight_availability(nearby airport)")
    else:
        print("Thought: flight is available, proceeding to book hotel next.")
        print("Action: book hotel, then plan itinerary")

random.seed(1)   # deterministic for this demo run
plan_then_execute("Coorg")
print()
interleaved_planning("Coorg")


--- Plan-then-execute ---
Full plan committed up front: ['Book flight', 'Book hotel', 'Plan itinerary']
Executing 'Book flight'... available=False
  Flight unavailable, but the plan didn't account for this -- proceeding anyway (this is the brittleness of committing to a full plan up front).
Executing 'Book hotel'...
Executing 'Plan itinerary'...

--- Interleaved planning (ReAct-style) ---
Thought: first step should be booking a flight. Let's check availability.
Observation: flight available = True
Thought: flight is available, proceeding to book hotel next.
Action: book hotel, then plan itinerary


**Exercise 4.4:** Run the cell above with a few different `random.seed(...)`
values (or remove the seed entirely) until you get a run where the flight is
*unavailable*. Compare how each strategy handled it. Which one produced a
more sensible response to the surprise? What did that robustness cost (in
general -- think about how many "Thought" steps interleaved planning needs
compared to plan-then-execute)?


Comparison of the two strategies
1. Plan-then-execute: The agent creates the entire plan upfront: book a flight, book a hotel, and plan the itinerary. If the flight is unavailable, it continues with the original plan anyway, even though the remaining steps may no longer make sense.
2. Interleaved planning (ReAct-style): The agent checks flight availability first and observes the result. If the flight is unavailable, it considers alternatives, such as trying a nearby airport or a different date, before booking a hotel.
Which is more sensible?
Interleaved planning is more sensible because it adapts to unexpected situations instead of blindly following a fixed plan. It avoids wasting effort booking a hotel before confirming that travel is possible.
What does this robustness cost?
Interleaved planning generally requires more reasoning steps because the agent repeatedly observes results, evaluates the situation, and decides what to do next. This can increase execution time, API calls, and computational cost compared with plan-then-execute.
Conclusion: Plan-then-execute is simpler and often faster when conditions are predictable. Interleaved planning is more flexible and reliable when unexpected changes can affect later steps.

## 4. Self-reflection / self-correction (Reflexion-style)

Instead of retraining a model when it fails, have it generate a *verbal*
critique of its own failed attempt, and carry that critique forward as
context for the next attempt.


In [4]:
def attempt_task(task: str, memory_of_past_reflections: list) -> str:
    """# TODO: replace with a real LLM call that uses `memory_of_past_reflections`
    as additional context to inform this attempt.
    """
    if not memory_of_past_reflections:
        return f"(stub) First attempt at '{task}': used a generic approach."
    return (f"(stub) Attempt at '{task}', informed by prior reflection: "
            f"'{memory_of_past_reflections[-1]}'")

def evaluate(attempt: str) -> bool:
    """# TODO: replace with a real LLM call (or real test/grader) that
    judges whether the attempt succeeded."""
    return "informed by prior reflection" in attempt   # succeeds only after 1 reflection

def self_reflect_on_failure(task: str, failed_attempt: str) -> str:
    """# TODO: replace with a real LLM call that critiques `failed_attempt`
    and proposes a concrete improvement."""
    return f"The generic approach didn't address the specifics of '{task}' -- be more specific next time."

def reflexion_loop(task: str, max_attempts: int = 3):
    reflections = []
    for i in range(max_attempts):
        attempt = attempt_task(task, reflections)
        print(f"Attempt {i + 1}: {attempt}")
        if evaluate(attempt):
            print("Evaluator: success!")
            return attempt
        reflection = self_reflect_on_failure(task, attempt)
        print(f"Self-reflection: {reflection}")
        reflections.append(reflection)
    print("Max attempts reached without success.")
    return attempt

reflexion_loop("write a personalized welcome email")


Attempt 1: (stub) First attempt at 'write a personalized welcome email': used a generic approach.
Self-reflection: The generic approach didn't address the specifics of 'write a personalized welcome email' -- be more specific next time.
Attempt 2: (stub) Attempt at 'write a personalized welcome email', informed by prior reflection: 'The generic approach didn't address the specifics of 'write a personalized welcome email' -- be more specific next time.'
Evaluator: success!


"(stub) Attempt at 'write a personalized welcome email', informed by prior reflection: 'The generic approach didn't address the specifics of 'write a personalized welcome email' -- be more specific next time.'"

**Exercise 4.5 (mini deliverable):** Notice `evaluate()` here is a hard-coded
rule that only exists to make the demo terminate cleanly. In a real system,
what could serve as the evaluator? Consider at least two options: (a) a
second, separate LLM call acting as a judge, and (b) something *not* based
on an LLM at all (e.g., a unit test, a validation rule, a user's explicit
feedback). Give one example task where each type of evaluator would be more
appropriate, and explain why.

**Exercise 4.6:** Connect this notebook's Part 4 back to Day 3's
evaluator-optimizer workflow pattern. What's the same? What's different
about *who* is doing the evaluating and reflecting?


Exercise 4.5: Possible evaluators in a real system
In a real system, the evaluator checks whether an attempt meets the task's requirements. It can use a second LLM call or a non-LLM method.
(a) A separate LLM acting as a judge
A second LLM evaluates the generated output against criteria such as relevance, clarity, correctness, and tone.
- Example task: Writing a personalized welcome email.
- How it works: The judge checks whether the email uses the recipient's name, matches the requested tone, includes the necessary information, and avoids irrelevant content.
- Why appropriate: Writing quality and personalization are difficult to evaluate using simple rules alone.
(b) A non-LLM evaluator
A traditional program, unit test, validation rule, or explicit user feedback checks whether the result meets specific requirements.
- Example task: Calculating the total price of items in a shopping cart.
- How it works: A unit test compares the calculated total against the expected value for a known set of items.
- Why appropriate: Mathematical calculations have precise expected results, so a deterministic test is generally more reliable and cheaper than asking another LLM to judge them.
Other examples include checking whether an email address has a valid format or asking a user whether a generated response solved their problem.

Exercise 4.6: Connection to the evaluator-optimizer workflow
What is the same?
Both workflows generate an initial result, evaluate it, identify shortcomings, and use feedback to improve the next attempt. This cycle continues until the result meets the success criteria or a maximum number of attempts is reached.
What is different?
- Evaluator-optimizer workflow (Day 3): An evaluator assesses the output and provides feedback to an optimizer, which revises the result. The evaluator and optimizer are distinct logical roles, even if they use the same underlying LLM.
- Reflexion workflow (Part 4): The agent evaluates its failed attempt, generates a natural-language reflection about what went wrong, and stores that reflection in memory. The next attempt uses this stored experience to avoid repeating the same mistake.
Example: When writing a welcome email, an evaluator-optimizer might revise the email based on feedback that its tone is too formal. A Reflexion agent might additionally remember, "The previous attempt sounded generic because it did not use the recipient's details," and use that lesson when generating the next version.
Conclusion: Both workflows use feedback to improve outputs, but Reflexion emphasizes learning from previous attempts through stored reflections that influence future behavior. In the provided notebook, this memory is only a list of reflections within the current run, not persistent memory across separate sessions.